# Stage 2: choosing the matches

**What this stage does:** keeps only the matches that belong to our study.

A match stays in if it is a **men's international** match, **England** is playing, the format is **Test, ODI
or T20I**, and the **first day** of the match is between 1 January 2020 and 29 October 2026. Every match that
is left out gets one reason written next to it.

**Reads:** the unzipped match files
**Writes:** `data/interim/file_audit.csv` (every file with its decision) and `data/processed/match_metadata.csv`
(only the matches we keep)

In [1]:
%run ./library/lib_00_settings.ipynb
%run ./library/lib_01_data_loading.ipynb
stage_log = get_stage_logger("02_filter")

In [2]:
file_audit = audit_match_files(list_match_files(extract_archive_if_needed()))
write_table(file_audit, INTERIM_FOLDER / "file_audit.csv")

included_matches = file_audit[file_audit["included"]].drop(columns=["included", "exclusion_reason"])
write_table(included_matches, PROCESSED_FOLDER / "match_metadata.csv")

### What we kept, and why the rest was left out

In [3]:
stage_log.info("files checked: %d, matches kept: %d", len(file_audit), len(included_matches))
stage_log.info("reasons for leaving files out:\n%s",
               file_audit.loc[~file_audit["included"], "exclusion_reason"].value_counts().to_string())
stage_log.info("matches kept per format:\n%s", included_matches["format"].value_counts().reindex(list(FORMATS)).to_string())
stage_log.info("first and last match day: %s .. %s", included_matches["start_date"].min(), included_matches["start_date"].max())

# A few limited-overs matches were washed out before England batted. They are part of the study but have no
# England innings, so they won't produce any partnerships. We list them so nobody is surprised later.
matches_without_england_innings = included_matches[included_matches["n_innings_england"] == 0]
stage_log.info("kept matches where England did not bat: %d %s",
               len(matches_without_england_innings), matches_without_england_innings["match_id"].tolist())

2026-11-19 18:05:02,449 INFO partnerships.02_filter: files checked: 961, matches kept: 287


2026-11-19 18:05:02,451 INFO partnerships.02_filter: reasons for leaving files out:
exclusion_reason
before_window    674


2026-11-19 18:05:02,452 INFO partnerships.02_filter: matches kept per format:
format
Test     81
ODI      88
T20I    118


2026-11-19 18:05:02,452 INFO partnerships.02_filter: first and last match day: 2020-01-03 .. 2026-10-29


2026-11-19 18:05:02,453 INFO partnerships.02_filter: kept matches where England did not bat: 7 ['1185311', '1239536', '1276912', '1415706', '1433377', '1491719', '1496574']
